In [ ]:
#importing the libraries needed
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
#downloading the data set
# 1. Download the data set
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

#data normalization
# Converting the data type to 32-bit floats instead of the defulat value of 64-bit float to save GPU memory and speed up training time
train_images = train_images.astype('float32')
test_images = test_images.astype('float32')

train_images, test_images = train_images / 255.0, test_images / 255.0
#this normalizes pixel values to be between 0 and 1



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_images,train_labels,test_size=0.2)
#splitting training and testing data sets

In [ ]:
len (X_train)


40000

In [ ]:
len (X_test)

10000

In [ ]:
len (y_train)

40000

In [ ]:
len (y_test)

10000

In [ ]:
print("training label" ,train_images.shape)
print("train label :", train_labels.shape)
print("test label :", test_images.shape)
# this is to check that the data is loaded correctly, the 50000 is the total number of images, and the 32 means that each image is 32x32 pixels, and the 3 shows that there are 3 color channels, RGB( red, green, bluee), and the one in the training label is the id assigned to each image.

training label (50000, 32, 32, 3)
train label : (50000, 1)
test label : (10000, 32, 32, 3)


In [ ]:
#data augmentaiton
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])


train_images_augmented = data_augmentation(train_images)

In [ ]:
#traning models

from tensorflow.keras.applications import VGG16
#training from scratch
model_a_scratch = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])
model_a_scratch.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])



model_a_scratch.summary()
#transfer model

base_model_transfer = VGG16(include_top=False, input_shape=(96, 96, 3), weights='imagenet')
base_model_transfer.trainable = True

model_a_transfer = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(96, 96),
    base_model_transfer,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])
model_a_transfer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                         loss='sparse_categorical_crossentropy',
                         metrics=['accuracy'])
model_a_transfer.summary()



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 592,554 (2.26 MB)

 Trainable params: 591,914 (2.26 MB)

 Non-trainable params: 640 (2.50 KB)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 3, 3, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 14,848,586 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
  #callbacks and early stopping cell

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=6)
my_callbacks = [early_stop, reduce_lr]
# Scratch Model
print("Training Model A (Scratch)...")
history_a_scratch = model_a_scratch.fit(
    train_images, train_labels,
    epochs=50,
    batch_size=64,
    validation_split=0.3,
    callbacks=my_callbacks
)
#Transfer Model
print("\nTraining Model A (Transfer)...")
history_a_transfer = model_a_transfer.fit(
    train_images, train_labels,
    epochs=50,
    batch_size=64,
    validation_split=0.3,
    callbacks=my_callbacks
)



Training Model A (Scratch)...
Epoch 1/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 23s 24ms/step - accuracy: 0.4656 - loss: 1.5568 - val_accuracy: 0.4752 - val_loss: 1.4226 - learning_rate: 0.0010
Epoch 2/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6322 - loss: 1.0474 - val_accuracy: 0.6693 - val_loss: 0.9563 - learning_rate: 0.0010
Epoch 3/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.6843 - loss: 0.8986 - val_accuracy: 0.7157 - val_loss: 0.8185 - learning_rate: 0.0010
Epoch 4/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7142 - loss: 0.8114 - val_accuracy: 0.7216 - val_loss: 0.8015 - learning_rate: 0.0010
Epoch 5/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7418 - loss: 0.7476 - val_accuracy: 0.7386 - val_loss: 0.7499 - learning_rate: 0.0010
Epoch 6/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7564 - loss: 0.6999 - val_accuracy: 0.7453 - val_loss: 0.7277 - learning_rate: 0.0010
Epoch 7/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - a

In [ ]:

print("Scratch Model Evaluation:")
model_a_scratch.evaluate(test_images, test_labels)

print("\nTransfer Model Evaluation:")
model_a_transfer.evaluate(test_images, test_labels)


Final Results
Scratch Model Evaluation:
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8087 - loss: 0.5965

Transfer Model Evaluation:
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 31ms/step - accuracy: 0.9159 - loss: 0.2548


[0.2548491954803467, 0.9158999919891357]